In [4]:
"""
Représentation des Immune Microenvironments (immune_ME) par patient ordonnés par GFR croissant
"""
import sys, io

# Fix encodage UTF-8 uniquement si on est dans un vrai terminal (pas Jupyter/VS Code notebook)
if hasattr(sys.stdout, "buffer"):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

out = Path(r"C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR")
out.mkdir(parents=True, exist_ok=True)

# ── Charger ─────────────────────────────────────────────────────────────────
print("Chargement...")

adata = ad.read_h5ad(r"C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\adata_spatial_final.h5ad")

diag = pd.read_csv(r"C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\Diagnosis.csv", sep=";")
diag["GFR"] = diag["GFR"].astype(str).str.strip()

def base_id(s):
    s = str(s).strip()
    for suf in ["_Xenium", "_CosMx"]:
        if s.endswith(suf):
            s = s[:-len(suf)]
    return s

diag["orig_ident"] = diag["Sample ID"].apply(base_id)
diag_unique = diag.drop_duplicates("orig_ident")[["orig_ident","GFR","Condition"]].copy()
diag_unique = diag_unique[diag_unique["GFR"].notna() & (diag_unique["GFR"] != "nan")]

# Proportions immune_ME par patient
meta = adata.obs[["orig_ident","immune_ME"]].copy()
meta["orig_ident"] = meta["orig_ident"].astype(str)

counts = (meta.groupby(["orig_ident","immune_ME"], observed=True)
              .size().unstack(fill_value=0))
props  = counts.div(counts.sum(axis=1), axis=0)
props.columns = props.columns.astype(str)
props.index   = props.index.astype(str)

diag_idx = diag_unique.set_index("orig_ident")[["GFR","Condition"]]
diag_idx.index = diag_idx.index.astype(str)
props_gfr = props.join(diag_idx, how="inner")

print(f"Patients avec GFR + immune_ME: {len(props_gfr)}")

gfr_order  = ["<30", "30-60", ">60"]
gfr_colors = {"<30": "#d6604d", "30-60": "#f4a582", ">60": "#4393c3"}
gfr_labels = {"<30": "GFR < 30\n(sévère)", "30-60": "GFR 30–60\n(modéré)", ">60": "GFR > 60\n(préservé)"}

props_gfr["GFR_cat"] = pd.Categorical(props_gfr["GFR"], categories=gfr_order, ordered=True)
props_gfr = props_gfr.sort_values(["GFR_cat","Condition"])

me_cols = [c for c in props_gfr.columns if c not in ["GFR","Condition","GFR_cat"]]

# Palette immune ME
palette = {
    "Unknown":                   "#cccccc",
    "Immune Immune 1":           "#e41a1c",
    "Residential Immune ME":     "#377eb8",
    "Inj. Tubular Immune ME":    "#ff7f00",
    "Fibro Immune ME":           "#984ea3",
    "Vascular Immune ME":        "#a65628",
    "Glomerular Immune ME":      "#4daf4a",
    "B predom. Immune ME":       "#f781bf",
}
me_colors = {me: palette.get(me, "#888888") for me in me_cols}

# ── 1. Stacked bar par GFR ──────────────────────────────────────────────────
# ── 1. Stacked bar par GFR ──────────────────────────────────────────────────
print("Figure 1: stacked bar...")
n_pat = len(props_gfr)
fig, ax = plt.subplots(figsize=(max(14, n_pat * 0.38), 7))
x = np.arange(n_pat)
bottom = np.zeros(n_pat)

for me in me_cols:
    vals = props_gfr[me].values
    ax.bar(x, vals, bottom=bottom, color=me_colors[me],
           label=me, width=0.85, edgecolor="none")
    bottom += vals

prev = 0
for gfr in gfr_order:
    n = (props_gfr["GFR_cat"] == gfr).sum()
    if n == 0:
        prev += n; continue
    if prev > 0:
        ax.axvline(prev - 0.5, color="black", lw=1.5, ls="--")
    ax.text(prev + n / 2, 1.035, gfr_labels[gfr],
            ha="center", fontsize=10, fontweight="bold",
            color=gfr_colors[gfr], transform=ax.get_xaxis_transform())
    ax.axvspan(prev - 0.5, prev + n - 0.5, ymin=0, ymax=0.02,
               color=gfr_colors[gfr], alpha=0.5, zorder=5)
    prev += n

ax.set_xticks(x)
ax.set_xticklabels(props_gfr.index, rotation=65, ha="right", fontsize=7)
for tick, pat in zip(ax.get_xticklabels(), props_gfr.index):
    tick.set_color(gfr_colors.get(props_gfr.loc[pat,"GFR"], "black"))
for xi, pat in enumerate(props_gfr.index):
    ax.text(xi, -0.06, props_gfr.loc[pat,"Condition"], rotation=65, ha="right",
            fontsize=5.5, color="grey", transform=ax.get_xaxis_transform())

ax.set_ylabel("Proportion de cellules", fontsize=12)
ax.set_ylim(0, 1)
ax.set_title("Immune Microenvironments par patient ordonnés par GFR croissant", fontsize=13)
ax.legend(handles=[mpatches.Patch(color=me_colors[m], label=m) for m in me_cols],
          bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9, title="Immune ME")

# ── Flèche/bande dégradée GFR croissant (style triangle, au-dessus du plot) ──
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# Créer un axe supplémentaire au-dessus du plot principal, même largeur en x
ax_gfr = ax.inset_axes([0, 1.08, 1, 0.08], transform=ax.transAxes)
ax_gfr.set_xlim(ax.get_xlim())
ax_gfr.axis("off")

# Dégradé vert clair -> vert foncé (faible GFR -> fort GFR, gauche à droite)
cmap_gfr = LinearSegmentedColormap.from_list("gfr_grad", ["#d9f0d3", "#1b7837"])

n_segments = 200
x_min, x_max = ax.get_xlim()
xs = np.linspace(x_min, x_max, n_segments + 1)

# Forme triangulaire : hauteur croissante de gauche à droite (comme une flèche pleine)
y_top_start, y_top_end = 0.25, 1.0   # hauteur relative du triangle (0 à 1 dans ax_gfr)

for i in range(n_segments):
    frac = i / n_segments
    y_top = y_top_start + frac * (y_top_end - y_top_start)
    color = cmap_gfr(frac)
    ax_gfr.fill_between(
        [xs[i], xs[i + 1]], [0, 0], [y_top, y_top],
        color=color, edgecolor="none", linewidth=0
    )

ax_gfr.set_ylim(0, 1)
ax_gfr.text(x_max, 0.5, " GFR", ha="left", va="center",
            fontsize=11, fontweight="bold", color="#1b7837",
            transform=ax_gfr.transData)

fig.tight_layout()
fig.savefig(out / "01_immuneME_by_GFR_stacked.png", dpi=150, bbox_inches="tight")
plt.close()
print("  saved 01_immuneME_by_GFR_stacked.png")

# ── 2. Heatmap proportion moyenne ME × GFR ──────────────────────────────────
print("Figure 2: heatmap...")
mean_by_gfr = props_gfr.groupby("GFR_cat", observed=True)[me_cols].mean()
mean_by_gfr.index = [gfr_labels[g] for g in mean_by_gfr.index]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(mean_by_gfr, cmap="YlOrRd", ax=ax, annot=True, fmt=".2f",
            annot_kws={"size": 9}, linewidths=0.5,
            cbar_kws={"label": "Proportion moyenne"})
ax.set_xlabel("Immune Microenvironment", fontsize=11)
ax.set_ylabel("Catégorie GFR", fontsize=11)
ax.set_title("Proportion moyenne des Immune ME par catégorie de GFR", fontsize=13)
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
fig.tight_layout()
fig.savefig(out / "02_immuneME_heatmap_mean_by_GFR.png", dpi=150, bbox_inches="tight")
plt.close()
print("  saved 02_immuneME_heatmap_mean_by_GFR.png")

# ── 3. Boxplot par catégorie GFR ────────────────────────────────────────────
print("Figure 3: boxplot...")
n_me = len(me_cols)
ncols = 4
nrows = int(np.ceil(n_me / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.2))
axes = axes.flatten()

for i, me in enumerate(me_cols):
    ax = axes[i]
    groups = [props_gfr.loc[props_gfr["GFR_cat"] == g, me].dropna()
              for g in gfr_order if (props_gfr["GFR_cat"] == g).sum() > 0]
    labels_g = [gfr_labels[g].replace("\n"," ")
                for g in gfr_order if (props_gfr["GFR_cat"] == g).sum() > 0]
    colors_g = [gfr_colors[g]
                for g in gfr_order if (props_gfr["GFR_cat"] == g).sum() > 0]

    bp = ax.boxplot(groups, tick_labels=labels_g, patch_artist=True,
                    medianprops=dict(color="black", lw=2), widths=0.5)
    for patch, col in zip(bp["boxes"], colors_g):
        patch.set_facecolor(col); patch.set_alpha(0.6)
    for j, (grp, col) in enumerate(zip(groups, colors_g)):
        jitter = np.random.uniform(-0.1, 0.1, len(grp))
        ax.scatter(j + 1 + jitter, grp, color=col, s=20, alpha=0.8, zorder=3)

    if len(groups) == 3:
        _, p = stats.kruskal(*groups); test = "KW"
    elif len(groups) == 2:
        _, p = stats.mannwhitneyu(*groups, alternative="two-sided"); test = "MW"
    else:
        p = np.nan; test = ""
    stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    ax.set_title(f"{me}\n{stars} p={p:.3f} ({test})" if not np.isnan(p) else me, fontsize=8)
    ax.set_ylabel("Proportion", fontsize=7)
    ax.tick_params(labelsize=6.5)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Proportion des Immune ME par catégorie de GFR", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(out / "03_immuneME_boxplot_by_GFR.png", dpi=150, bbox_inches="tight")
plt.close()
print("  saved 03_immuneME_boxplot_by_GFR.png")

# ── 4. Corrélations Spearman ME ~ GFR numérique ─────────────────────────────
print("Figure 4: corrélations Spearman...")
gfr_num = {"<30": 15, "30-60": 45, ">60": 75}
props_gfr["GFR_num"] = props_gfr["GFR"].map(gfr_num)

corr_results = []
for me in me_cols:
    sub = props_gfr[[me,"GFR_num"]].dropna()
    if len(sub) > 3:
        r, p = stats.spearmanr(sub["GFR_num"], sub[me])
        stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        corr_results.append({"immune_ME": me, "spearman_r": r, "pval": p, "sig": stars})

corr_df = pd.DataFrame(corr_results).sort_values("spearman_r")
corr_df.to_csv(out / "spearman_immuneME_GFR.csv", index=False)

print("\n  Corrélations significatives (p<0.05):")
print(corr_df[corr_df["pval"] < 0.05].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [gfr_colors["<30"] if r < 0 else gfr_colors[">60"]
              for r in corr_df["spearman_r"]]
bars = ax.barh(corr_df["immune_ME"], corr_df["spearman_r"],
               color=colors_bar, alpha=0.85)
for bar, row in zip(bars, corr_df.itertuples()):
    if row.pval < 0.05:
        offset = 0.01 if row.spearman_r >= 0 else -0.01
        ax.text(row.spearman_r + offset, bar.get_y() + bar.get_height() / 2,
                row.sig, va="center",
                ha="left" if row.spearman_r >= 0 else "right",
                fontsize=10, fontweight="bold")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Spearman r (Immune ME ~ GFR)", fontsize=11)
ax.set_title("Corrélation Immune ME ~ GFR\n(bleu = augmente avec GFR, rouge = diminue avec GFR)",
             fontsize=12)
ax.tick_params(labelsize=9)
fig.tight_layout()
fig.savefig(out / "04_spearman_immuneME_GFR.png", dpi=150, bbox_inches="tight")
plt.close()
print("  saved 04_spearman_immuneME_GFR.png")

# ── 5. Scatter proportion ~ GFR pour chaque ME ──────────────────────────────
print("Figure 5: scatter...")
ncols = 4
nrows = int(np.ceil(n_me / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
axes = axes.flatten()

for i, me in enumerate(me_cols):
    ax = axes[i]
    sub = props_gfr[[me,"GFR_num","GFR"]].dropna()
    for gfr_cat, col in gfr_colors.items():
        mask = sub["GFR"] == gfr_cat
        ax.scatter(sub.loc[mask,"GFR_num"] + np.random.uniform(-2,2,mask.sum()),
                   sub.loc[mask, me], color=col, s=35, alpha=0.85,
                   label=gfr_cat, zorder=3)
    if len(sub) > 3:
        z = np.polyfit(sub["GFR_num"], sub[me], 1)
        xr = np.linspace(10, 80, 100)
        ax.plot(xr, np.poly1d(z)(xr), "k--", lw=1.2, alpha=0.7)
        r, p = stats.spearmanr(sub["GFR_num"], sub[me])
        stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        ax.set_title(f"{me}\nr={r:.2f}, {stars}", fontsize=8)
    else:
        ax.set_title(me, fontsize=8)
    ax.set_xticks([15, 45, 75])
    ax.set_xticklabels(["<30","30-60",">60"], fontsize=7)
    ax.set_xlabel("GFR", fontsize=7)
    ax.set_ylabel("Proportion", fontsize=7)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7, title="GFR", loc="upper right")
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Proportion des Immune ME ~ GFR (Spearman)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(out / "05_immuneME_scatter_GFR.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nTermine. Figures dans {out}/")

Chargement...
Patients avec GFR + immune_ME: 57
Figure 1: stacked bar...
  saved 01_immuneME_by_GFR_stacked.png
Figure 2: heatmap...
  saved 02_immuneME_heatmap_mean_by_GFR.png
Figure 3: boxplot...
  saved 03_immuneME_boxplot_by_GFR.png
Figure 4: corrélations Spearman...

  Corrélations significatives (p<0.05):
             immune_ME  spearman_r         pval sig
       Immune Immune 1   -0.651442 4.072963e-08 ***
Inj. Tubular Immune ME   -0.493083 9.744014e-05 ***
       Fibro Immune ME   -0.356385 6.508309e-03  **
    Vascular Immune ME    0.432908 7.698492e-04 ***
  Glomerular Immune ME    0.467289 2.476714e-04 ***
 Residential Immune ME    0.633519 1.229211e-07 ***
  saved 04_spearman_immuneME_GFR.png
Figure 5: scatter...

Termine. Figures dans C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR/


In [7]:
# ── 1. Stacked bar par GFR ──────────────────────────────────────────────────
print("Figure 1: stacked bar...")
n_pat = len(props_gfr)
fig, ax = plt.subplots(figsize=(max(16, n_pat * 0.4), 8))
x = np.arange(n_pat)
bottom = np.zeros(n_pat)

for me in me_cols:
    vals = props_gfr[me].values
    ax.bar(x, vals, bottom=bottom, color=me_colors[me],
           label=me, width=0.85, edgecolor="none")
    bottom += vals

prev = 0
for gfr in gfr_order:
    n = (props_gfr["GFR_cat"] == gfr).sum()
    if n == 0:
        prev += n; continue
    ax.text(prev + n / 2, 1.035, gfr_labels[gfr],
            ha="center", fontsize=10, fontweight="bold",
            color=gfr_colors[gfr], transform=ax.get_xaxis_transform())
    ax.axvspan(prev - 0.5, prev + n - 0.5, ymin=0, ymax=0.02,
               color=gfr_colors[gfr], alpha=0.5, zorder=5)
    prev += n

# ── Labels patient seul, en noir, sans lignes de séparation ─────────────────
ax.set_xticks(x)
ax.set_xticklabels(props_gfr.index, rotation=65, ha="right", fontsize=7, color="black")

ax.set_ylabel("Proportion de cellules", fontsize=12)
ax.set_ylim(0, 1)
ax.set_title("Immune Microenvironments par patient ordonnés par GFR croissant", fontsize=13, pad=70)
ax.legend(handles=[mpatches.Patch(color=me_colors[m], label=m) for m in me_cols],
          bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9, title="Immune ME")

# ── Flèche/bande dégradée GFR croissant ──────────────────────────────────────
from matplotlib.colors import LinearSegmentedColormap

ax_gfr = ax.inset_axes([0, 1.12, 1, 0.06], transform=ax.transAxes)
ax_gfr.set_xlim(ax.get_xlim())
ax_gfr.axis("off")

cmap_gfr = LinearSegmentedColormap.from_list("gfr_grad", ["#d9f0d3", "#1b7837"])

n_segments = 200
x_min, x_max = ax.get_xlim()
xs = np.linspace(x_min, x_max, n_segments + 1)

y_top_start, y_top_end = 0.25, 1.0

for i in range(n_segments):
    frac = i / n_segments
    y_top = y_top_start + frac * (y_top_end - y_top_start)
    color = cmap_gfr(frac)
    ax_gfr.fill_between(
        [xs[i], xs[i + 1]], [0, 0], [y_top, y_top],
        color=color, edgecolor="none", linewidth=0
    )

ax_gfr.set_ylim(0, 1)
ax_gfr.text(x_max, 0.5, " GFR", ha="left", va="center",
            fontsize=11, fontweight="bold", color="#1b7837",
            transform=ax_gfr.transData)

fig.tight_layout()
fig.savefig(out / "01_immuneME_by_GFR_stacked.png", dpi=150, bbox_inches="tight")
plt.close()
print("  saved 01_immuneME_by_GFR_stacked.png")

Figure 1: stacked bar...
  saved 01_immuneME_by_GFR_stacked.png


In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

BASE = Path(r"C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR")
out = BASE / "figures" / "spatial_celltype_abundance"
out.mkdir(parents=True, exist_ok=True)

# ── 1. Colonnes des types cellulaires déconvolués ────────────────────────────
celltype_cols = [
    'B_Cells', 'B_Memory', 'B_Naive', 'Basophile', 'CD14_Mono', 'CD16_Mono',
    'CD4_Activated', 'CD4_Trm', 'CD4_signaling', 'CD8_MAIT', 'CD8_central_memory',
    'CD8_cytotoxic/effector_memory', 'FOLR2+_resident', 'FOLR2_CKD', 'NK/T_cells',
    'NK_cytotoxic', 'Neutro_FPR2+', 'Plasma_cells', 'TREM2+_macro', 'cDC', 'pDC'
]
celltype_cols = [c for c in celltype_cols if c in adata.obs.columns]

# ── 2. Choisir un échantillon représentatif par technologie ─────────────────
np.random.seed(42)
sample_per_tech = {}
for t in ['Xenium', 'CosMx']:
    samples_t = adata.obs.loc[adata.obs['tech'] == t, 'orig_ident'].unique()
    if len(samples_t) == 0:
        print(f"Aucun échantillon trouvé pour {t}")
        continue
    chosen = np.random.choice(samples_t)
    sample_per_tech[t] = chosen
    n_cells = ((adata.obs['tech'] == t) & (adata.obs['orig_ident'] == chosen)).sum()
    print(f"{t}: échantillon choisi = {chosen} (n = {n_cells:,} cellules)")

# ── 3. Plot : un panel par technologie, coloré par cellule dominante ────────
dominant_type = adata.obs[celltype_cols].idxmax(axis=1)
adata.obs['dominant_celltype'] = pd.Categorical(dominant_type)

cell_types_order = adata.obs['dominant_celltype'].cat.categories.tolist()
colors = plt.cm.tab20.colors
palette_dict = {ct: colors[i % 20] for i, ct in enumerate(cell_types_order)}

fig, axes = plt.subplots(1, len(sample_per_tech), figsize=(8 * len(sample_per_tech), 8))
if len(sample_per_tech) == 1:
    axes = [axes]

for ax, (t, sample_id) in zip(axes, sample_per_tech.items()):
    mask = (adata.obs['tech'] == t) & (adata.obs['orig_ident'] == sample_id)
    sub_coords = adata.obsm['spatial'][mask.values]
    sub_labels = adata.obs.loc[mask, 'dominant_celltype']

    for ct in cell_types_order:
        ct_mask = sub_labels == ct
        if ct_mask.sum() == 0:
            continue
        ax.scatter(
            sub_coords[ct_mask.values, 0],
            sub_coords[ct_mask.values, 1],
            c=[palette_dict[ct]],
            s=3, alpha=0.8, label=ct, rasterized=True
        )

    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title(f"{t} — {sample_id}\n(n = {mask.sum():,} cellules)", fontsize=12, fontweight='bold')

handles = [mpatches.Patch(color=palette_dict[ct], label=ct) for ct in cell_types_order]
fig.legend(handles=handles, bbox_to_anchor=(1.0, 0.5), loc='center left',
           fontsize=8, frameon=False, title='Type dominant (cell2location)')

plt.tight_layout()
plt.savefig(out / "spatial_dominant_celltype_xenium_cosmx.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure sauvegardée : {out / 'spatial_dominant_celltype_xenium_cosmx.png'}")

# ── 4. Alternative : carte de densité continue pour UN type cellulaire ──────
target_celltype = 'FOLR2_CKD'

fig, axes = plt.subplots(1, len(sample_per_tech), figsize=(8 * len(sample_per_tech), 8))
if len(sample_per_tech) == 1:
    axes = [axes]

for ax, (t, sample_id) in zip(axes, sample_per_tech.items()):
    mask = (adata.obs['tech'] == t) & (adata.obs['orig_ident'] == sample_id)
    sub_coords = adata.obsm['spatial'][mask.values]
    sub_score = adata.obs.loc[mask, target_celltype].values

    sca = ax.scatter(
        sub_coords[:, 0], sub_coords[:, 1],
        c=sub_score, cmap='viridis', s=3, alpha=0.85, rasterized=True
    )
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title(f"{t} — {sample_id}\n{target_celltype} abundance", fontsize=12, fontweight='bold')
    plt.colorbar(sca, ax=ax, shrink=0.6, label='Abondance')

plt.tight_layout()
fig_path = out / f"spatial_{target_celltype.replace('/','_')}_abundance.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure sauvegardée : {fig_path}")

Xenium: échantillon choisi = 1008 (n = 3,158 cellules)
CosMx: échantillon choisi = HK3626 (n = 2,318 cellules)


C:\Users\RICHMOND\AppData\Local\Temp\ipykernel_28932\1581970253.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure sauvegardée : C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR\figures\spatial_celltype_abundance\spatial_dominant_celltype_xenium_cosmx.png
Figure sauvegardée : C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR\figures\spatial_celltype_abundance\spatial_FOLR2_CKD_abundance.png


C:\Users\RICHMOND\AppData\Local\Temp\ipykernel_28932\1581970253.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

out = Path(r"C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR")
out.mkdir(parents=True, exist_ok=True)

# ── 1. Colonnes des types cellulaires déconvolués ────────────────────────────
celltype_cols = [
    'B_Cells', 'B_Memory', 'B_Naive', 'Basophile', 'CD14_Mono', 'CD16_Mono',
    'CD4_Activated', 'CD4_Trm', 'CD4_signaling', 'CD8_MAIT', 'CD8_central_memory',
    'CD8_cytotoxic/effector_memory', 'FOLR2+_resident', 'FOLR2_CKD', 'NK/T_cells',
    'NK_cytotoxic', 'Neutro_FPR2+', 'Plasma_cells', 'TREM2+_macro', 'cDC', 'pDC'
]
celltype_cols = [c for c in celltype_cols if c in adata.obs.columns]

# ── 2. Type dominant par cellule (idxmax) ────────────────────────────────────
adata.obs['dominant_celltype'] = pd.Categorical(adata.obs[celltype_cols].idxmax(axis=1))

# ── 3. Vérifications ──────────────────────────────────────────────────────────
print("Immune_ME_20um catégories:")
print(adata.obs['Immune_ME_20um'].value_counts())
print("\nNaN dans Immune_ME_20um:", adata.obs['Immune_ME_20um'].isna().sum())

# ── 4. Table croisée : proportion de chaque type dominant dans chaque ME ────
crosstab = pd.crosstab(
    adata.obs['Immune_ME_20um'],
    adata.obs['dominant_celltype'],
    normalize='index'   # proportion par ligne (par ME)
)

crosstab.to_csv(out / "dominant_celltype_by_immuneME_proportions.csv")
print(f"\nTable sauvegardée : {out / 'dominant_celltype_by_immuneME_proportions.csv'}")

# ── 5. Stacked bar : composition cellulaire de chaque immune ME ─────────────
cell_types_order = crosstab.columns.tolist()
colors = plt.cm.tab20.colors
palette_dict = {ct: colors[i % 20] for i, ct in enumerate(cell_types_order)}

fig, ax = plt.subplots(figsize=(max(10, len(crosstab.index) * 1.2), 7))
bottom = np.zeros(len(crosstab))
x = np.arange(len(crosstab))

for ct in cell_types_order:
    vals = crosstab[ct].values
    ax.bar(x, vals, bottom=bottom, color=palette_dict[ct], label=ct, width=0.7, edgecolor='white', linewidth=0.3)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(crosstab.index, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Proportion (type dominant)', fontsize=12)
ax.set_xlabel('Immune Microenvironment (20µm)', fontsize=12)
ax.set_title('Composition cellulaire dominante par Immune ME', fontsize=13)
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, frameon=False, title='Type cellulaire dominant')

plt.tight_layout()
plt.savefig(out / "dominant_celltype_by_immuneME_stackedbar.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure sauvegardée : {out / 'dominant_celltype_by_immuneME_stackedbar.png'}")

# ── 6. Heatmap (vue alternative, plus lisible si beaucoup de catégories) ────
fig, ax = plt.subplots(figsize=(max(10, len(cell_types_order) * 0.6), max(5, len(crosstab.index) * 0.6)))
sns.heatmap(
    crosstab, cmap='YlOrRd', annot=True, fmt='.2f',
    annot_kws={'size': 7}, linewidths=0.5,
    cbar_kws={'label': 'Proportion'}, ax=ax
)
ax.set_xlabel('Type cellulaire dominant', fontsize=11)
ax.set_ylabel('Immune Microenvironment (20µm)', fontsize=11)
ax.set_title('Proportion du type dominant par Immune ME', fontsize=13)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig(out / "dominant_celltype_by_immuneME_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure sauvegardée : {out / 'dominant_celltype_by_immuneME_heatmap.png'}")

Immune_ME_20um catégories:
Immune_ME_20um
Immune Immune 1           63695
Residential Immune ME     60501
Inj. Tubular Immune ME    48471
Fibro Immune ME           38825
Unknown                   14015
Vascular Immune ME        12698
Glomerular Immune ME      12503
B predom. Immune ME        3307
Name: count, dtype: int64

NaN dans Immune_ME_20um: 0

Table sauvegardée : C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR\dominant_celltype_by_immuneME_proportions.csv


C:\Users\RICHMOND\AppData\Local\Temp\ipykernel_28932\3242971344.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figure sauvegardée : C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR\dominant_celltype_by_immuneME_stackedbar.png
Figure sauvegardée : C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\xemuin\cell2location_output\plots\immuneME_by_GFR\dominant_celltype_by_immuneME_heatmap.png


C:\Users\RICHMOND\AppData\Local\Temp\ipykernel_28932\3242971344.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
